# Lab 33: Graph RAG from scratch

Build a knowledge graph from Lab 06's corpus, detect communities, summarize them, and answer global (map-reduce) and local (traversal) questions. Fill in the `TODO` cells; reference implementation in `solution/`.

Pattern source: Edge et al. 2024 ([arXiv:2404.16130](https://arxiv.org/abs/2404.16130)).

Note: we use networkx greedy modularity as a stand-in for the paper's Leiden algorithm — faithful to the control flow, simplified in the clustering.

New dependency: `uv add 'networkx>=3.0'`

## Step 0: Setup

One new dependency beyond Lab 06: `networkx` for the graph and community detection.

```bash
uv add 'networkx>=3.0'
```

In [ ]:
import hashlib
import json
import os
import pathlib
import re
from typing import Any
from collections import defaultdict
from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")
PROVIDER = "openai"
MODEL = {"openai": "gpt-4o-mini", "anthropic": "claude-haiku-4-5-20251001"}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")

In [ ]:
def chat(messages: list[dict], temperature: float = 0.0) -> str:
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=MODEL, messages=messages, temperature=temperature)
        return resp.choices[0].message.content or ""
    else:
        from anthropic import Anthropic
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        resp = Anthropic().messages.create(
            model=MODEL, system=system, messages=non_system,
            max_tokens=1500, temperature=temperature)
        return "".join(b.text for b in resp.content if hasattr(b, "text"))


def chat_json(messages):
    raw = chat(messages).strip()
    raw = re.sub(r"^```(json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"(\{.*\}|\[.*\])", raw, re.DOTALL)
        return json.loads(m.group(0)) if m else {}

## Step 1: Load documents

GraphRAG extracts entities at the document level, not the fine retrieval-chunk level, so we keep whole docs here. The corpus is this lab's own entity-rich `./corpus/` (a fictional research ecosystem), not Lab 06's concept corpus.

In [ ]:
CORPUS_DIR = pathlib.Path("./corpus")
def split_paras(t): return [p.strip() for p in re.split(r"\n\s*\n", t) if p.strip()]

# GraphRAG works at the document/section level for entity extraction, not the
# fine retrieval chunks. We read each doc and keep its sections.
docs = []
for path in sorted(CORPUS_DIR.glob("*.md")):
    if path.name == "README.md":
        continue
    body = path.read_text()
    title = body.splitlines()[0].lstrip("# ").strip()
    docs.append({"doc_id": path.stem, "title": title, "text": body})
print(f"Loaded {len(docs)} documents for graph construction")

## Step 2: Extract entities and relationships

The expensive index-time step: one LLM call per document. This cost is why GraphRAG suits stable corpora — re-indexing on every corpus change is pricey.

In [ ]:
def extract_entities_relations(doc: dict) -> dict:
    """TODO: prompt the model to extract entities and relationships as JSON:
    {"entities":[{"name","type"}], "relations":[{"source","target","desc"}]}.
    Use chat_json(...). Set defaults for missing keys. Ask for canonical names."""
    raise NotImplementedError

## Step 3: Build the knowledge graph

Nodes are entities (tracking which docs they came from); edges are relationships with descriptions.

In [ ]:
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

def build_graph(docs: list[dict]) -> nx.Graph:
    """TODO: for each doc call extract_entities_relations; add entity nodes
    (track which docs they appear in via a set attr) and relationship edges
    (store desc). Lowercase names so the same concept merges. Return G."""
    raise NotImplementedError

G = build_graph(docs)
print(f"Graph: {G.number_of_nodes()} entities, {G.number_of_edges()} relationships")

## Step 4: Detect communities

Cluster related entities. The paper uses Leiden; we use networkx greedy modularity as a lab-grade stand-in (marked).

In [ ]:
def detect_communities(G: nx.Graph) -> list[set]:
    """Index-time step 3: detect communities of related entities. The GraphRAG
    paper uses the Leiden algorithm; we use networkx greedy modularity, which is
    a reasonable stand-in for a lab (Leiden needs the igraph/leidenalg packages).
    Marked as a simplification, not a reproduction."""
    if G.number_of_edges() == 0:
        return [set(G.nodes())]
    return list(greedy_modularity_communities(G))

communities = detect_communities(G)
print(f"Detected {len(communities)} communities")
for i, c in enumerate(communities):
    print(f"  community {i}: {sorted(c)[:6]}{' ...' if len(c) > 6 else ''}")

## Step 5: Summarize each community

These summaries are what a GLOBAL query maps over — the mechanism that lets GraphRAG answer corpus-wide questions flat RAG cannot.

In [ ]:
def summarize_community(G: nx.Graph, community: set) -> str:
    """Index-time step 4: summarize each community. These summaries are what a
    GLOBAL query maps over. We feed the LLM the entities and their internal
    relationships."""
    entities = sorted(community)
    edges = [(u, v, G.edges[u, v].get("desc", ""))
             for u, v in G.subgraph(community).edges()]
    edge_lines = "\n".join(f"- {u} -> {v}: {d}" for u, v, d in edges) or "(no internal edges)"
    return chat([
        {"role": "system", "content": "Summarize what this cluster of related "
         "concepts is about in 2-3 sentences."},
        {"role": "user", "content":
         f"Entities: {', '.join(entities)}\n\nRelationships:\n{edge_lines}"},
    ])

community_summaries = [
    {"id": i, "entities": sorted(c), "summary": summarize_community(G, c)}
    for i, c in enumerate(communities)
]
for cs in community_summaries:
    print(f"[community {cs['id']}] {cs['summary'][:100]}...")

## Step 6: Query — global (map-reduce) and local (traversal)

Global questions map over community summaries then reduce; local questions traverse the neighborhood of the entities the query mentions.

In [ ]:
def graph_rag_global(query: str) -> str:
    """TODO (map-reduce):
      map:    for each community summary, ask for a partial answer (or 'NONE')
      reduce: synthesize the non-NONE partials into one answer, citing community ids
    """
    raise NotImplementedError

def graph_rag_local(query: str, hops: int = 1) -> str:
    """TODO: find seed entities mentioned in the query, expand to neighbors within
    `hops`, build the subgraph edge list, and answer from those relationships."""
    raise NotImplementedError

## Step 7: Route and answer

A cheap classifier picks global vs local.

In [ ]:
def graph_rag(query: str) -> dict:
    """Route global vs local by a cheap classification, then answer."""
    kind = chat([
        {"role": "system", "content": "Is this a GLOBAL question (about overall "
         "themes/patterns across the whole corpus) or a LOCAL question (about a "
         "specific entity or relationship)? Answer one word: global or local."},
        {"role": "user", "content": query},
    ]).strip().lower()
    if "global" in kind:
        return {"query": query, "route": "global", "answer": graph_rag_global(query)}
    return {"query": query, "route": "local", "answer": graph_rag_local(query)}

## Step 8: See the global-question win

The payoff: a 'what are the themes' question that flat chunk retrieval handles poorly, because the answer is not in any single chunk — it is spread across the whole corpus.

In [ ]:
# GraphRAG\'s signature win is the GLOBAL question that flat chunk-retrieval
# handles poorly, because the answer requires synthesizing across the whole
# corpus rather than retrieving a few similar chunks.
print("=== GLOBAL question (GraphRAG\'s strength) ===")
g = graph_rag("What are the main themes connecting the documents in this corpus?")
print(f"  route={g['route']}")
print(f"  {g['answer'][:300]}\n")

print("=== LOCAL question (specific entity) ===")
local_result = graph_rag("What is the agent loop related to?")
print(f"  route={local_result['route']}")
print(f"  {local_result['answer'][:240]}")

## What you built

A from-scratch GraphRAG pipeline: entity/relationship extraction, graph construction, community detection, community summarization, and both global (map-reduce) and local (traversal) query paths. The high index-time cost (an LLM call per document, plus summarization per community) is the defining tradeoff — GraphRAG earns it for stable corpora with rich entity structure and global/multi-hop queries.

**Where this implementation simplifies:** networkx greedy modularity stands in for Leiden; community summarization is flat rather than hierarchical; entity-name canonicalization is best-effort (lowercasing) rather than entity resolution. Each is a reasonable next extension and is marked in the code.

See [`concepts/rag/sota-rag-patterns.md`](../../concepts/rag/sota-rag-patterns.md) (Pattern 5) and [`diagrams/rag-bundle.md#6-graph-rag-workflow`](../../diagrams/rag-bundle.md).